In [1]:
import numpy as np


In [2]:
class Exponential:
    '''
    Convenience class for the exponential distribution.
    packages up distribution parameters, seed and random generator.
    '''
    def __init__(self, mean, random_seed=None):
        '''
        Constructor
        
        Params:
        ------
        mean: float
            The mean of the exponential distribution
        
        random_seed: int, optional (default=None)
            A random seed to reproduce samples.  If set to none then a unique
            sample is created.
        '''
        self.rand = np.random.default_rng(seed=random_seed)
        self.mean = mean
        
    def sample(self, size=None):
        '''
        Generate a sample from the exponential distribution
        
        Params:
        -------
        size: int, optional (default=None)
            the number of samples to return.  If size=None then a single
            sample is returned.
        '''
        return self.rand.exponential(self.mean, size=size)



class Lognormal:
    """
    Encapsulates a lognormal distirbution
    """
    def __init__(self, mean, stdev, random_seed=None):
        """
        Params:
        -------
        mean = mean of the lognormal distribution
        stdev = standard dev of the lognormal distribution
        """
        self.rand = np.random.default_rng(seed=random_seed)
        mu, sigma = self.normal_moments_from_lognormal(mean, stdev**2)
        self.mu = mu
        self.sigma = sigma
        
    def normal_moments_from_lognormal(self, m, v):
        '''
        Returns mu and sigma of normal distribution
        underlying a lognormal with mean m and variance v
        source: https://blogs.sas.com/content/iml/2014/06/04/simulate-lognormal
        -data-with-specified-mean-and-variance.html

        Params:
        -------
        m = mean of lognormal distribution
        v = variance of lognormal distribution
                
        Returns:
        -------
        (float, float)
        '''
        phi = math.sqrt(v + m**2)
        mu = math.log(m**2/phi)
        sigma = math.sqrt(math.log(phi**2/m**2))
        return mu, sigma
        
    def sample(self):
        """
        Sample from the normal distribution
        """
        return self.rand.lognormal(self.mu, self.sigma)


class Normal:
    
    def __init__(self, mean, std, random_seed=None):
        self.rand = np.random.default_rng(seed=random_seed)
        self.mean = mean
        self.std = std
        
    def sample(self, size=None):
        return self.rand.normal(loc=self.mean, scale=self.std, size=size)

In [3]:
# These are the parameters for the model

# resource counts
N_BED = 24


# time between arrivals in hours (exponential)
Emergency_IAT = 37.03
Ward_IAT = 25.97
Other_IAT = 47.1
AnE_IAT = 22.72
Xray_IAT = 575.09

# mean inter-arrival time in hours (Normal distribution)
Elective_mean_IAT = 19.51
# standard deviation of arrival times
Elective_std_IAT = 26.42

# mean Length of stay (Lognormal)
mean_Emergency_LOS = 140.15
mean_Ward_LOS = 177.89
mean_Other_LOS = 212.86
mean_AnE_LOS = 128.86
mean_Xray_LOS = 87.53
mean_Elective_LOS = 57.34


# Standard deviation of length of stay (Lognormal)
std_Emergency_LOS = 218.02
std_Ward_LOS = 276.54
std_Other_LOS = 457.67
std_AnE_LOS = 267.51
std_Xray_LOS = 108.15
std_Elective_LOS = 99.78


# SEEDS to reproduce results of a single run
REPRODUCIBLE_RUN = True
    
if REPRODUCIBLE_RUN:
    SEEDS = [42, 101, 1066, 1966, 2013, 999, 1444, 2016, 1456, 56,78,456]
else:
    SEEDS = [None, None, None, None, None, None, None, None]

In [6]:

class Scenario:
    '''
    Parameter container class for the Critical Care Unit (CCU) model.

    This class encapsulates all resource capacities, arrival rates, and 
    probability distribution parameters for a single simulation run. It utilizes 
    the provided global constants as defaults but allows for granular 
    parameterization of experimental scenarios via the constructor.

    Attributes
    ----------
    name : str or None
        A descriptive name for the scenario.
    N_bed : int
        The number of available triage bay resources.
    arrival_dist_emerg : Exponential
        Distribution for inter-arrival times from emergency
    arrival_dist_ward :  Exponential
        Distribution for inter-arrival times from the ward
    arrival_dist_other : Exponential
        Distribution for inter-arrival times from other sources
    arrival_dist_AnE : Exponential
        Distribution for inter-arrival times from A/E
    arrival_dist_xray : Exponential
        Distribution for inter-arrival times from Xray
    arrival_dist_elective : Normal Distribution
        Distribution for inter-arrival times.
    emerg_dist : Lognormal
        Distribution for the length of stay for emergency patients
    ward_dist : Lognormal
        Distribution for the length of stay for ward patients
    other_dist : Lognormal
        Distribution for the length of stay for ward patients
    AnE_dist : Lognormal
        Distribution for the length of stay for ward patients
    xray_dist : Lognormal
        Distribution for the length of stay for ward patients
    elect_dist : Lognormal
        Distribution for the length of stay for ward patients
    
    '''
    def __init__(
        self, 
        name=None,
        # Resources
        CCU_beds = N_BED,
        # Arrival parameters for Unplanned patients
        emerg_iat = Emergency_IAT,
        ward_iat = Ward_IAT,
        other_iat = Other_IAT,
        AnE_iat = AnE_IAT,
        xray_iat = Xray_IAT ,
        # Arrival parameters for Unplanned patients
        elective_mean_iat = Elective_mean_IAT,
        elective_std_iat = Elective_std_IAT,
        # LOS parameters for unplanned patients
        # Emergency
          emerg_mean_los = mean_Emergency_LOS,
          emerg_std_los = std_Emergency_LOS,
        # Ward 
           ward_mean_los = mean_Ward_LOS ,
           ward_std_los = std_Ward_LOS,
        # Other
           other_mean_los = mean_Other_LOS,
           other_std_los = std_Other_LOS,
        # Xray
           xray_mean_los = mean_Xray_LOS,
           xray_std_los = std_Xray_LOS,
        # AnE
           AnE_mean_los = mean_AnE_LOS,
           AnE_std_los = std_AnE_LOS,
        # LOS parameters for unplanned patients
        # Elective
           elect_mean_los = mean_Elective_LOS,
           elect_mean_std = std_Elective_LOS,
        # Random Seeds
        seeds=SEEDS
    ):
        '''
        The init method sets up our defaults using the global constants.
        Any parameter can be overridden by passing a value to the constructor.
        
        Params:
        -------
        name : str or None
            optional name for scenario
        '''
        # optional name
        self.name = name
        
        # triage bays
        self.CCU_beds = N_BED
        
        # inter-arrival distribution for unplanned cases
        self.arrival_dist_emerg = Exponential( Emergency_IAT, random_seed=seeds[0])
        self.arrival_dist_ward = Exponential( Ward_IAT, random_seed=seeds[1])
        self.arrival_dist_other = Exponential(Other_IAT, random_seed=seeds[2])
        self.arrival_dist_AnE = Exponential( AnE_IAT, random_seed=seeds[3])
        self.arrival_dist_xray = Exponential( Xray_IAT , random_seed=seeds[4])

       # inter-arrival distribution for planned cases
        self.arrival_dist_elective = Normal(elective_mean_iat,elective_std_iat, random_seed=seeds[5])

       #length of stay distribution for unplanned patients
        self.emerg_dist = Lognormal(emerg_mean_los, emerg_std_los, 
                                         random_seed=seeds[6])
        self.ward_dist = Lognormal(ward_mean_los, ward_std_los, 
                                         random_seed=seeds[7])
        self.other_dist = Lognormal(other_mean_los, other_std_los, 
                                         random_seed=seeds[8])
        self.AnE_dist = Lognormal(AnE_mean_los, AnE_std_los, 
                                         random_seed=seeds[9])
        self.xray_dist = Lognormal(xray_mean_los, xray_std_los, 
                                         random_seed=seeds[10])
        self.elect_dist = Lognormal(elect_mean_los, elect_std_los, 
                                         random_seed=seeds[11])
